# Logistic Regression - Hyperparameter Tuning & Evaluation

**Dataset:** UNSW-NB15 (Binary: Normal vs Attack)

**Approach:**
1. Manual C sweep (course pattern) to visualize regularization effect
2. RandomizedSearchCV for broad hyperparameter exploration
3. GridSearchCV to refine around best region
4. Threshold tuning to maximize F1/recall (originality)
5. Final evaluation on held-out test set

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from model_benchmarking.logistic_regression import (
    load_data, sweep_c_values, plot_c_sweep,
    run_randomized_search, get_refined_grid, run_grid_search,
    evaluate_model, find_best_threshold, plot_threshold_analysis,
    plot_confusion_matrix, plot_roc_curve, plot_precision_recall_curve,
    plot_search_results, save_model, save_results,
    RANDOM_STATE
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Load Data
If preprocessed CSVs don't exist, regenerate from raw data.

In [2]:
train_path = '../data/processed/training_lr_mlp.csv'
test_path = '../data/processed/test_lr_mlp.csv'

# Regenerate if CSVs don't exist
if not os.path.exists(train_path) or not os.path.exists(test_path):
    print("Preprocessed CSVs not found. Running preprocessing pipeline...")
    from preprocessing.preprocessing import preprocess_train, preprocess_test, save_artifacts
    
    raw_train = pd.read_csv('../data/raw/UNSW_NB15_training-set.csv')
    raw_test = pd.read_csv('../data/raw/UNSW_NB15_testing-set.csv')
    
    df_full, df_model, df_model_tree, artifacts = preprocess_train(raw_train)
    test_full, test_model, test_model_tree = preprocess_test(raw_test, artifacts)
    
    df_model.to_csv(train_path, index=False)
    test_model.to_csv(test_path, index=False)
    save_artifacts(artifacts, '../data/processed/artifacts.pkl')
    print("Preprocessing complete.")

X_train, y_train, X_test, y_test = load_data(train_path, test_path)
print(f"Train shape: {X_train.shape}")
print(f"Test shape:  {X_test.shape}")
print(f"\nLabel distribution (train):")
print(y_train.value_counts())

Preprocessed CSVs not found. Running preprocessing pipeline...


FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/UNSW_NB15_training-set.csv'

## 2. C Sweep (Course Pattern)
Manual loop over C values with log scale, tracking train vs validation accuracy.

In [ ]:
# Split training into train/val for manual sweep
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE
)
print(f"Train subset: {X_tr.shape}, Val subset: {X_val.shape}")

In [ ]:
sweep_df = sweep_c_values(X_tr, y_tr, X_val, y_val)
sweep_df

In [ ]:
fig = plot_c_sweep(sweep_df)
plt.show()

## 3. RandomizedSearchCV (Broad Exploration)
Search over C, penalty (l1/l2/elasticnet), class_weight, and l1_ratio.

In [ ]:
random_search = run_randomized_search(X_train, y_train, n_iter=50, cv=5)
print(f"\nBest params: {random_search.best_params_}")
print(f"Best CV score: {random_search.best_score_:.4f}")

In [ ]:
# Top 10 results from random search
top_results = plot_search_results(random_search, n_top=10)
top_results

## 4. GridSearchCV (Refinement)
Build a tight grid around the best region found by random search.

In [ ]:
param_grid = get_refined_grid(random_search)
print("Refined grid:")
for k, v in param_grid.items():
    print(f"  {k}: {v}")

In [ ]:
grid_search = run_grid_search(X_train, y_train, param_grid, cv=5)
print(f"\nBest params: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

In [ ]:
# Top grid search results
grid_results = plot_search_results(grid_search, n_top=10)
grid_results

## 5. Threshold Tuning (Originality)
Instead of using the default 0.5 threshold, find the threshold that maximizes F1.
For intrusion detection, missing an attack (false negative) is worse than a false alarm.

In [ ]:
best_model = grid_search.best_estimator_

# Find best threshold on validation set
best_threshold_f1, threshold_df = find_best_threshold(best_model, X_val, y_val, metric="f1")
best_threshold_recall, _ = find_best_threshold(best_model, X_val, y_val, metric="recall")

print(f"Best threshold (maximize F1):     {best_threshold_f1:.2f}")
print(f"Best threshold (maximize Recall):  {best_threshold_recall:.2f}")
print(f"Default threshold:                 0.50")

In [ ]:
fig = plot_threshold_analysis(threshold_df)
plt.show()

## 6. Final Evaluation on Test Set
Compare default threshold (0.5) vs tuned threshold.

In [ ]:
# Evaluate with default threshold
results_default = evaluate_model(best_model, X_test, y_test, threshold=0.5)
print("=" * 60)
print("EVALUATION WITH DEFAULT THRESHOLD (0.5)")
print("=" * 60)
print(results_default['classification_report_text'])
print(f"Accuracy:  {results_default['accuracy']:.4f}")
print(f"F1 Score:  {results_default['f1']:.4f}")
print(f"Recall:    {results_default['recall']:.4f}")
print(f"ROC AUC:   {results_default['roc_auc']:.4f}")

In [ ]:
# Evaluate with tuned threshold
results_tuned = evaluate_model(best_model, X_test, y_test, threshold=best_threshold_f1)
print("=" * 60)
print(f"EVALUATION WITH TUNED THRESHOLD ({best_threshold_f1:.2f})")
print("=" * 60)
print(results_tuned['classification_report_text'])
print(f"Accuracy:  {results_tuned['accuracy']:.4f}")
print(f"F1 Score:  {results_tuned['f1']:.4f}")
print(f"Recall:    {results_tuned['recall']:.4f}")
print(f"ROC AUC:   {results_tuned['roc_auc']:.4f}")

In [ ]:
# Comparison summary
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "F1", "Recall", "ROC AUC"],
    "Default (0.5)": [
        results_default['accuracy'], results_default['f1'],
        results_default['recall'], results_default['roc_auc']
    ],
    f"Tuned ({best_threshold_f1:.2f})": [
        results_tuned['accuracy'], results_tuned['f1'],
        results_tuned['recall'], results_tuned['roc_auc']
    ],
})
comparison["Difference"] = comparison[f"Tuned ({best_threshold_f1:.2f})"] - comparison["Default (0.5)"]
comparison

## 7. Plots

In [ ]:
fig = plot_confusion_matrix(results_default, title="Confusion Matrix (Default Threshold 0.5)")
plt.show()

In [ ]:
fig = plot_confusion_matrix(results_tuned, title=f"Confusion Matrix (Tuned Threshold {best_threshold_f1:.2f})")
plt.show()

In [ ]:
fig = plot_roc_curve(results_default)
plt.show()

In [ ]:
fig = plot_precision_recall_curve(results_default)
plt.show()

## 8. Save Best Model & Results

In [ ]:
save_model(best_model, '../models/logistic_regression/best_model.pkl')
save_results(results_tuned, '../models/logistic_regression/results.pkl')

print(f"\nBest hyperparameters: {grid_search.best_params_}")
print(f"Best threshold: {best_threshold_f1:.2f}")
print(f"Test F1 (tuned): {results_tuned['f1']:.4f}")